# 참고 실행 코드
이 파일은 이전에 제공한 실험 노트북을 저장된 실험 설정에 맞춘 참고본입니다. 실제 Colab에서 실행한 원본 사본은 제공되지 않았으므로 코드의 완전 일치를 보증하지 않습니다. 결과 수치는 results/window_compare_02에서 확인하세요.


# Window 크기 비교 실험 — PyTorch
원본의 환경·LSTM AE·전처리를 재사용한 독립 실행 노트북입니다. Colab T4에서 위부터 실행하고 원본 CSV 두 개를 업로드하세요. 과거 출력은 제거했습니다. 실제 실험 결과는 실행 후 생성됩니다.

**빠른 점검:** 아래 `EXPERIMENT_EPOCHS=2`, `SEEDS=[42]`. **본 실험:** 800 epoch, seed 42/43/44. 점검 결과로 성능 결론을 내리지 마세요.

비교 설정은 5-A절의 한 셀에서 바꿉니다. 원본과 달리 모든 window에서 입력 마지막 행·라벨 행·표본 수를 고정하고 Valid/Test 입력 중복을 제거합니다. 그래서 원본 20 결과와 수치가 같지 않습니다. 여기서 20도 다시 학습합니다.

### 원본에서 직접 한 번씩 바꾸는 경우
|위치|수정/처리|
|---|---|
|2절 `CONFIG = dict(sequence=20, ...)`|20을 10/30 등으로 변경하는 핵심 설정|
|2절 `RUN_ID`|`w10_s42`, `w30_s42`처럼 변경해 결과 덮어쓰기 방지|
|3절 환경 연결 이후|설정 셀 실행 후 여기부터 다시 실행해 별도 커널에 전달|
|7절 `make_windows`, `prepare`, `split_manifest`|`cfg.sequence`를 전달하므로 수정 불필요. 전처리 재실행 필요|
|8절 `LSTMAutoencoder`|`self.sequence`로 반복 길이를 정하므로 수정 불필요. 새 모델 학습 필요|
|10절 임계값·평가|새 모델로 다시 실행. 기존 임계값 재사용 금지|
|5절 Config 기본값, 7절 make_windows 기본값의 20|명시한 CONFIG 값이 우선하므로 중복 수정 불필요|
|7절 expected shape·설명, 8절 RepeatVector(20) 설명|shape 검사는 기본 설정일 때만 실행. 설명의 고정 숫자는 새 길이에 맞춰 해석|

원본 방식의 크기별 표본 수(나머지 설정 기본값):
|window|Train|Valid|Test 정상|Test 이상|Test 합계|
|---:|---:|---:|---:|---:|---:|
|10|14890|1180|4010|190|4200|
|20|14880|1180|4000|180|4180|
|30|14870|1180|3990|170|4160|

본 비교 방식의 기본 `ALIGN_WINDOW=50`에서는 모든 후보가 Train 14850, Valid 1180, Test 4022(정상 3921, 이상 101)입니다. Test 이상이 적고 중첩 표본이므로 독립 사건 101개를 뜻하지 않습니다.

## 1. Colab 환경 준비

PyTorch 2.7.0의 CUDA 12.6 배포본을 사용합니다. GPU 라이브러리는 TensorFlow 2.7 환경과 다릅니다. [공식 설치 버전 안내](https://pytorch.org/get-started/previous-versions/). 가이드에서 버전을 지정하지 않은 분석 패키지는 호환되는 버전으로 고정합니다.

In [ ]:
# 모든 설치는 현재 Colab 서버의 /content 안에서 이루어집니다.
# Colab 기본 Python/패키지를 교체하지 않고 가이드용 Python 3.9.7을 실행합니다.
import os, sys, json, subprocess, urllib.request, tarfile, io
from pathlib import Path
try:
    from google.colab import files
except ImportError:
    raise RuntimeError("이 노트북은 Google Colab에서 열고 실행하세요.")
if sys.platform != "linux":
    raise RuntimeError("Colab의 호스팅 런타임을 선택하세요.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("런타임 → 런타임 유형 변경에서 T4 GPU를 선택하세요.")
print("Colab GPU:", gpu.stdout.strip())
print("Colab 기본 Python:", sys.version.split()[0], "→ 학습은 별도 Python 3.9.7에서 실행")

FRAMEWORK_NAME = "pytorch"
ENV_DIR = Path("/content/kamp_guide_" + FRAMEWORK_NAME)
MAMBA = Path("/content/kamp_tools/micromamba")
PYTHON = ENV_DIR / "bin/python"
if not MAMBA.is_file():
    MAMBA.parent.mkdir(parents=True, exist_ok=True)
    url = "https://github.com/mamba-org/micromamba-releases/releases/download/2.3.3-0/micromamba-linux-64"
    print("Colab에 환경 설치 도구 준비 중...")
    with urllib.request.urlopen(url, timeout=60) as response:
        MAMBA.write_bytes(response.read())
    MAMBA.chmod(0o755)

install_env = os.environ.copy()
install_env["MAMBA_ROOT_PREFIX"] = "/content/kamp_mamba"
install_env["PYTHONNOUSERSITE"] = "1"
# 가이드가 명시한 버전: Python 3.9.7, NumPy 1.20.0, TensorFlow 2.7.0.
# 버전이 적혀 있지 않은 분석 패키지는 해당 시기와 호환되는 버전으로 고정합니다.
PACKAGES = {'numpy': '1.20.0', 'pandas': '1.3.5', 'scikit-learn': '1.0.2', 'scipy': '1.7.3', 'matplotlib': '3.5.3', 'seaborn': '0.11.2', 'joblib': '1.1.0', 'ipykernel': '6.16.2', 'ipython': '8.18.1', 'jupyter-client': '7.4.9', 'matplotlib-inline': '0.1.6', 'traitlets': '5.9.0', 'setuptools': '65.5.0', 'torch': '2.7.0+cu126'}
spec = {"python":"3.9.7", "packages":PACKAGES, "gpu_libraries":'PyTorch CUDA 12.6 wheel dependencies'}
marker = ENV_DIR / ".kamp_setup.json"
ready = marker.is_file() and json.loads(marker.read_text()) == spec
if not ready:
    subprocess.check_call([str(MAMBA), "create" if not PYTHON.exists() else "install",
        "-y", "-p", str(ENV_DIR), "--override-channels", "-c", "conda-forge",
        "python=3.9.7", "pip=24.3.1", *[]], env=install_env)
    requirements = [name + "==" + version for name, version in PACKAGES.items()]
    subprocess.check_call([str(PYTHON), "-m", "pip", "install", "--only-binary=:all:",
                           *requirements, *['--extra-index-url', 'https://download.pytorch.org/whl/cu126']], env=install_env)
    subprocess.check_call([str(PYTHON), "-m", "pip", "check"], env=install_env)
    marker.write_text(json.dumps(spec), encoding="utf-8")
print("가이드용 Colab 환경 준비 완료. 세션 재시작 없이 다음 셀로 진행하세요.")

## 2. 사용자 설정

In [ ]:
# 실행 설정: Colab 서버 경로이며 개인 PC/Drive ID가 필요 없습니다.
DATA_SOURCE = "upload"              # upload 또는 existing
DATA_DIR = "/content/kamp_data"
OUTPUT_ROOT = "/content/kamp_pytorch_results"
RUN_ID = "window_workspace"                    # 새 실험은 run02 등으로 변경
DOWNLOAD_RESULTS = True            # 마지막 셀에서 결과 ZIP 다운로드
REQUIRE_GPU = True

# 실험 기본 설정입니다.
CONFIG = dict(sequence=20, label_offset=100, train_rows=15000,
              valid_normal=880, valid_anomaly=300,
              epochs=800, batch_size=128, learning_rate=0.001,
              lr_factor=0.7, lr_patience=50,
              es_min_delta=0.00001, es_patience=120, seed=42)
# 처음 동작만 점검하려면 epochs=2, RUN_ID="smoke"로 바꿉니다.
# 설정을 바꾼 경우 환경 연결 셀부터 다시 실행하세요.

## 3. 가이드용 Python 실행 환경 연결

In [ ]:
# %%guide 셀을 Colab 서버 안의 Python 3.9.7에서 실행합니다.
# 셀 사이의 변수·모델은 유지되고, 표·그래프·학습 로그는 해당 셀 바로 아래에 표시됩니다.
import atexit
from jupyter_client import KernelManager
from jupyter_client.kernelspec import KernelSpecManager
from IPython.core.magic import register_cell_magic
from IPython.display import display, clear_output

if "guide_manager" in globals():
    guide_client.stop_channels()
    guide_manager.shutdown_kernel(now=True)
kernel_dir = Path("/content/kamp_kernels/guide")
kernel_dir.mkdir(parents=True, exist_ok=True)
(kernel_dir/"kernel.json").write_text(json.dumps({
    "argv":[str(PYTHON), "-m", "ipykernel_launcher", "-f", "{connection_file}",
            "--IPKernelApp.kernel_class=ipykernel.ipkernel.IPythonKernel",
            "--IPKernelApp.extensions=[]"],
    "display_name":"KAMP guide Python 3.9.7", "language":"python"
}), encoding="utf-8")
kernel_env = os.environ.copy()
kernel_env.pop("PYTHONPATH", None)
kernel_env.pop("PYTHONHOME", None)
kernel_env["PYTHONNOUSERSITE"] = "1"
kernel_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
kernel_env["IPYTHONDIR"] = str(ENV_DIR/"ipython")
kernel_env["JUPYTER_CONFIG_DIR"] = str(ENV_DIR/"jupyter_config")
kernel_env["PATH"] = str(ENV_DIR/"bin") + ":" + kernel_env.get("PATH", "")
kernel_env["LD_LIBRARY_PATH"] = str(ENV_DIR/"lib") + ":" + kernel_env.get("LD_LIBRARY_PATH", "")
kernel_env["MPLBACKEND"] = "module://matplotlib_inline.backend_inline"
kernel_env["TF_CPP_MIN_LOG_LEVEL"] = "2"
guide_manager = KernelManager(kernel_name="guide", kernel_spec_manager=KernelSpecManager(
    kernel_dirs=[str(kernel_dir.parent)]))
startup_log = open(ENV_DIR/"kernel_startup.log", "w")
guide_manager.start_kernel(cwd="/content", env=kernel_env, stdout=startup_log, stderr=startup_log)
guide_client = guide_manager.blocking_client()
guide_client.start_channels()
try:
    guide_client.wait_for_ready(timeout=120)
except Exception:
    startup_log.flush()
    print((ENV_DIR/"kernel_startup.log").read_text()[-6000:])
    raise

def show_guide_output(message):
    kind, content = message["header"]["msg_type"], message["content"]
    if kind == "stream":
        print(content["text"], end="", flush=True)
    elif kind in ("display_data", "execute_result"):
        display(content["data"], raw=True, metadata=content.get("metadata", {}))
    elif kind == "error":
        print("\n".join(content.get("traceback", [])))
    elif kind == "clear_output":
        clear_output(wait=content.get("wait", False))

@register_cell_magic
def guide(line, cell):
    try:
        reply = guide_client.execute_interactive(cell, timeout=None, allow_stdin=False,
                                                 output_hook=show_guide_output)
    except KeyboardInterrupt:
        guide_manager.interrupt_kernel()
        raise
    if reply["content"]["status"] != "ok":
        raise RuntimeError("가이드 실행 셀에서 오류가 발생했습니다. 위 오류 내용을 확인하세요.")

def close_guide_kernel():
    if guide_manager.has_kernel:
        guide_client.stop_channels()
        guide_manager.shutdown_kernel(now=True)
atexit.register(close_guide_kernel)
settings = {name:globals()[name] for name in (
    "DATA_SOURCE", "DATA_DIR", "OUTPUT_ROOT", "RUN_ID", "DOWNLOAD_RESULTS", "REQUIRE_GPU", "CONFIG")}
guide("", "import json\nglobals().update(json.loads(" + repr(json.dumps(settings)) + "))")
guide("", 'import sys\nprint("실제 분석·학습 Python:", sys.version.split()[0])\n'
          'get_ipython().run_line_magic("matplotlib", "inline")')

## 4. 데이터 업로드

처음 실행하면 CSV 두 개를 선택하는 창이 표시됩니다. 노트북에는 원본 CSV가 포함되어 있지 않으므로 실행자가 파일을 준비해야 합니다. `DATA_DIR`에 두 파일이 이미 있으면 업로드를 생략하고 해당 파일을 재사용합니다. 다른 데이터를 올리려면 설정 셀에서 `DATA_DIR`를 새 경로로 바꾸고 설정 셀부터 다시 실행하세요.

업로드는 빈 임시 폴더에서 받아 Colab의 중복 파일명 변경을 피합니다. 재실험 시에는 설정 셀의 `RUN_ID`도 새 이름으로 바꾸세요. 완료된 결과를 덮어쓰지 않도록 같은 `RUN_ID`의 재실행은 중단합니다.

In [ ]:
# CSV 업로드는 Colab 기본 환경에서, 분석은 가이드용 Python 환경에서 실행합니다.
import os
import tempfile

data_path = Path(DATA_DIR).expanduser().resolve()
data_path.mkdir(parents=True, exist_ok=True)
required_files = ("press_data_normal.csv", "outlier_data.csv")
if DATA_SOURCE not in ("upload", "existing"):
    raise ValueError("DATA_SOURCE는 upload 또는 existing입니다.")
if DATA_SOURCE == "upload" and not all((data_path/name).is_file() for name in required_files):
    print("원본 CSV 두 개를 함께 선택하세요:", ", ".join(required_files))
    # Colab이 중복 파일명에 (1)을 붙이지 않도록 매번 빈 임시 폴더에서 받습니다.
    previous_dir = Path.cwd()
    with tempfile.TemporaryDirectory(prefix="kamp_upload_", dir="/content") as upload_dir:
        try:
            os.chdir(upload_dir)
            uploaded = files.upload()
        finally:
            os.chdir(previous_dir)
        uploaded = {Path(name).name: contents for name, contents in uploaded.items()}
        missing = [name for name in required_files if name not in uploaded]
        if missing:
            raise ValueError("원본 파일명을 확인하고 CSV 두 개를 함께 선택하세요. 누락: " + ", ".join(missing))
        for name in required_files:
            (data_path/name).write_bytes(uploaded[name])
    del uploaded
    print("업로드한 CSV를 저장했습니다:", str(data_path))
else:
    print("Colab에 이미 있는 CSV를 사용합니다:", str(data_path))
if not all((data_path/name).is_file() for name in required_files):
    raise FileNotFoundError("DATA_DIR에 원본 CSV 두 개가 있는지 확인하세요.")
print("데이터 준비 완료:", ", ".join(required_files))

## 5. 설정과 실제 실행 환경

In [ ]:
%%guide
import os, sys, json, random, time, platform, subprocess, importlib.metadata
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display

FEATURES = ["AI0_Vibration", "AI1_Vibration", "AI2_Current"]
GUIDE_CM = np.array([[3922, 78], [26, 154]])
GUIDE = {"accuracy":4076/4180,"precision":154/232,"recall":154/180,
         "f1":308/412,"threshold":0.0034824829610499443,
         "validation_precision_recall":0.8466666666666667,
         "confusion_matrix":GUIDE_CM.tolist()}
@dataclass
class Config:
    sequence: int = 20
    label_offset: int = 100
    train_rows: int = 15000
    valid_normal: int = 880
    valid_anomaly: int = 300
    epochs: int = 800
    batch_size: int = 128
    learning_rate: float = 0.001
    lr_factor: float = 0.7
    lr_patience: int = 50
    es_min_delta: float = 0.00001
    es_patience: int = 120
    # Original guide does not set a seed; record one for repeatable experiments.
    seed: int = 42
def save_json(path, data):
    def convert(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, np.generic):
            return x.item()
        if isinstance(x, Path):
            return str(x)
        raise TypeError(type(x).__name__)
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=convert), encoding="utf-8")
cfg = Config(**CONFIG)
output = Path(OUTPUT_ROOT)/RUN_ID
if (output/"metrics.json").exists():
    raise FileExistsError("이 RUN_ID에는 완료된 결과가 있습니다. RUN_ID를 변경하고 다시 실행하세요.")
output.mkdir(parents=True, exist_ok=True)
random.seed(cfg.seed)
np.random.seed(cfg.seed)
save_json(output/"config.json", asdict(cfg))
display(pd.DataFrame(asdict(cfg).items(), columns=["설정", "값"]))

In [ ]:
%%guide
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True)
# TF 2.7의 TF32 허용 기본값에 맞춥니다. T4는 TF32를 지원하지 않습니다.
torch.backends.cudnn.allow_tf32=True
torch.backends.cuda.matmul.allow_tf32=True
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("PyTorch가 GPU를 인식하지 못했습니다. T4 런타임을 확인하세요.")
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
FRAMEWORK="PyTorch"
environment={"python":platform.python_version(),"framework":FRAMEWORK,"version":str(torch.__version__),
    "numpy":np.__version__,"pandas":pd.__version__,"scikit-learn":importlib.metadata.version("scikit-learn"),
    "gpu":torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU",
    "cuda":torch.version.cuda,"cudnn":torch.backends.cudnn.version(),"tf32_allowed":True}
save_json(output/"environment.json",environment)
display(pd.DataFrame(environment.items(),columns=["실행 환경","값"]))

## 5-A. 실험 설정
`WINDOW_SIZES`만 바꾸면 길이별 학습·평가를 반복합니다. `ALIGN_WINDOW=50`은 비교할 최대 길이이며 1차/2차 실험에서도 고정하세요. 이를 바꾸면 평가 표본이 바뀝니다.

1차: `[10,20,30] × [42,43,44]`. 2차: `[5,10,15,20,30,40,50]` 중 필요한 후보. 상한 50에서 가장 좋다면 별도 실험군으로 범위를 확장하고 20도 재측정하세요.

임계값 규칙은 **Valid F1 최대 + score >= threshold**입니다. 원본의 precision==recall / score>threshold와 다릅니다. 모든 후보에 같은 규칙을 적용합니다. 매번 정상 Valid MSE 기준의 최적 가중치를 복원합니다. 검증셋으로 임계값과 window를 함께 선택하므로 Valid 점수는 낙관적일 수 있습니다. Test로 후보를 선택하지 마세요.

In [ ]:
%%guide
WINDOW_SIZES = [10, 20, 30]
SEEDS = [42, 43, 44]
ALIGN_WINDOW = 50
EXPERIMENT_EPOCHS = 800
EXPERIMENT_ID = "window_compare_02"  # 설정 변경 시 새 이름
EXPERIMENT_DIR = Path(OUTPUT_ROOT) / EXPERIMENT_ID
assert len(set(WINDOW_SIZES)) == len(WINDOW_SIZES) and WINDOW_SIZES
assert len(set(SEEDS)) == len(SEEDS) and SEEDS
assert all(isinstance(w, int) and 1 <= w <= ALIGN_WINDOW for w in WINDOW_SIZES)
assert EXPERIMENT_EPOCHS > 0
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
experiment_spec = dict(config=CONFIG, windows=WINDOW_SIZES, seeds=SEEDS,
    align_window=ALIGN_WINDOW, epochs=EXPERIMENT_EPOCHS,
    threshold_rule="valid_max_f1_ge", split_rule="fixed_end_input_gap_v1",
    weights="best_normal_valid_mse_min_delta", environment=environment)
spec_path = EXPERIMENT_DIR / "experiment.json"
if spec_path.exists() and json.loads(spec_path.read_text()) != experiment_spec:
    raise ValueError("설정이 달라졌습니다. EXPERIMENT_ID를 새 이름으로 바꾸세요.")
save_json(spec_path, experiment_spec)

## 6. 데이터 로드와 공통 평가 시점 설정

In [ ]:
%%guide
def load_data(data_dir):
    normal = pd.read_csv(Path(data_dir) / "press_data_normal.csv", index_col=0)
    outlier = pd.read_csv(Path(data_dir) / "outlier_data.csv", index_col=0)
    for name, df, rows, state in [("normal", normal, 20000, 0), ("outlier", outlier, 600, 1)]:
        if list(df.columns) != ["TimeStamp"] + FEATURES + ["Equipment_state"]:
            raise ValueError(f"Unexpected {name} columns: {list(df.columns)}")
        if len(df) != rows or not df.Equipment_state.eq(state).all():
            raise ValueError(f"Unexpected {name} row count or labels")
        if df.isna().any().any() or not np.isfinite(df[FEATURES].to_numpy()).all():
            raise ValueError(f"Missing/non-finite data in {name}; do not silently change guidebook data")
    return normal, outlier
def quality_report(df, day):
    """Appendix p.56-63 formulas, plus an actual type check and duplicate diagnostics."""
    completeness = (1 - df.isna().mean()) * 100
    counts = df.value_counts()
    uniqueness = round((len(counts) - (counts > 1).sum()) / len(counts) * 100, 2)
    timestamps = pd.to_datetime(df.TimeStamp, errors="coerce")
    validity = timestamps.between(pd.Timestamp(day), pd.Timestamp(day) + pd.Timedelta(days=1), inclusive="left").mean() * 100
    consistency = 100.0  # literal appendix code: len(data) / len(data) * 100
    integrity = (1 - sum(x != 100 for x in [uniqueness, validity, consistency]) / 3) * 100
    dtypes_ok = all(str(df[c].dtype) == "float64" for c in FEATURES) and str(df.Equipment_state.dtype) == "int64"
    return {
        "rows": len(df), "missing_by_column": df.isna().sum().to_dict(),
        "completeness_by_column": completeness.to_dict(), "completeness": completeness.mean(),
        "uniqueness": uniqueness, "validity": validity, "consistency_guide_formula": consistency,
        "actual_numeric_dtypes_valid": dtypes_ok, "accuracy": None, "integrity": integrity,
        "weighted_quality": completeness.mean() * .6 + (uniqueness + validity + consistency + integrity) * .1,
        "duplicate_rows_after_first": int(df.duplicated().sum()),
        "duplicate_timestamps_after_first": int(timestamps.duplicated().sum()),
        "timestamp_monotonic": bool(timestamps.is_monotonic_increasing),
        "timestamp_step_seconds": timestamps.diff().dt.total_seconds().value_counts().head(10).to_dict(),
        "note": "Guide consistency formula always returns 100%; accuracy is not assessed. No rows removed.",
    }
def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=140)
    plt.show()
    plt.close()
def explore(normal, outlier, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    save_json(output / "data_quality.json", {
        "normal": quality_report(normal, "2022-07-12"),
        "outlier": quality_report(outlier, "2022-07-17"),
    })
    for name, df in [("normal", normal), ("outlier", outlier)]:
        df.head().to_csv(output / f"{name}_head.csv", encoding="utf-8-sig")
        df.describe().T.to_csv(output / f"{name}_describe.csv", encoding="utf-8-sig")
        for mode, data in [("raw", df[FEATURES]), ("absolute", df[FEATURES].abs())]:
            fig, axes = plt.subplots(3, 1, figsize=(16, 8))
            for col, ax in zip(FEATURES, axes):
                ax.plot(data[col], linewidth=.5)
                ax.set_title(f"{name}: {col} ({mode})")
            save_plot(output / f"{name}_{mode}.png")
        corr = df[FEATURES].abs().corr()
        corr.to_csv(output / f"{name}_correlation.csv")
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap="coolwarm")
        plt.title(f"{name}: correlation after abs()")
        save_plot(output / f"{name}_correlation.png")


normal,outlier=load_data(DATA_DIR)

In [ ]:
%%guide
import hashlib
fingerprints = {name: hashlib.sha256((Path(DATA_DIR)/name).read_bytes()).hexdigest()
                for name in ["press_data_normal.csv", "outlier_data.csv"]}
hash_path = EXPERIMENT_DIR / "data_sha256.json"
if hash_path.exists() and json.loads(hash_path.read_text()) != fingerprints:
    raise ValueError("CSV가 변경되었습니다. 새 EXPERIMENT_ID를 사용하세요.")
save_json(hash_path, fingerprints)

def prepare_aligned(normal, outlier, cfg, anchor):
    if not 1 <= cfg.sequence <= anchor or cfg.label_offset < 0:
        raise ValueError("sequence / anchor / offset 설정을 확인하세요.")
    scaler = MinMaxScaler().fit(normal[FEATURES].iloc[:cfg.train_rows].abs())
    segments = [("train", "normal", normal, 0, cfg.train_rows, 0),
                ("eval", "normal", normal, cfg.train_rows, len(normal), cfg.valid_normal),
                ("eval", "anomaly", outlier, 0, len(outlier), cfg.valid_anomaly)]
    xs = {k: [] for k in ["train", "valid", "test"]}
    ys = {k: [] for k in xs}
    records = []
    for kind, source, df, start, stop, nv in segments:
        values = scaler.transform(df[FEATURES].abs()).astype(np.float32)
        # 원본 라벨: end + offset + 1. 끝 행과 라벨 행은 window와 무관하게 고정.
        ends = np.arange(start + anchor - 1, stop - cfg.label_offset - 1)
        if kind == "train":
            groups = [("train", ends)]
        else:
            # 고정 anchor-1개를 경계에서 제외: 가장 긴 window도 입력 행이 겹치지 않음.
            if nv <= 0 or len(ends) <= nv + anchor - 1:
                raise ValueError("Valid/Test 표본이 부족합니다. 분할 또는 최대 길이를 검토하세요.")
            groups = [("valid", ends[:nv]), ("test", ends[nv + anchor - 1:])]
        for split, selected in groups:
            if not len(selected): raise ValueError("빈 split: " + split)
            labels = selected + cfg.label_offset + 1
            xs[split].append(np.stack([values[e-cfg.sequence+1:e+1] for e in selected]))
            ys[split].append(df.Equipment_state.to_numpy()[labels])
            records.append(pd.DataFrame(dict(split=split, source=source,
                input_start_row=selected-cfg.sequence+1, input_end_row=selected,
                label_row=labels, input_end_time=df.TimeStamp.iloc[selected].to_numpy(),
                label_time=df.TimeStamp.iloc[labels].to_numpy(),
                label=df.Equipment_state.iloc[labels].to_numpy())))
    data = {"scaler": scaler}
    for split in xs:
        data["X_"+split] = np.concatenate(xs[split])
        data["Y_"+split] = np.concatenate(ys[split])
    data["X_valid_0"] = data["X_valid"][data["Y_valid"] == 0]
    manifest = pd.concat(records, ignore_index=True)
    for source in ["normal", "anomaly"]:
        v = manifest[(manifest.source == source) & (manifest.split == "valid")]
        t = manifest[(manifest.source == source) & (manifest.split == "test")]
        assert v.input_end_row.max() < t.input_start_row.min()
    return data, manifest

preview_data, preview_manifest = prepare_aligned(normal, outlier, Config(**CONFIG), ALIGN_WINDOW)
display(preview_manifest.groupby(["split", "label"]).size().unstack(fill_value=0))
del preview_data

## 7. 모델과 학습 함수
원본 모델·optimizer·learning rate·조기 종료 조건을 유지합니다. 비교 시에는 최대 epoch 도달 여부와 관계없이 정상 Valid MSE의 최적 가중치를 사용합니다. `es_min_delta` 기준 개선 시 저장한 가중치이며 단순 최솟값과 다를 수 있습니다.

In [ ]:
%%guide
# TensorFlow 가이드의 LSTM AE 구조를 PyTorch로 옮깁니다.
class LSTMAutoencoder(nn.Module):
    def __init__(self, sequence):
        super().__init__()
        self.sequence=sequence
        self.encoder1 = nn.LSTM(3,64,batch_first=True)
        self.encoder2 = nn.LSTM(64,32,batch_first=True)
        self.decoder1 = nn.LSTM(32,32,batch_first=True)
        self.decoder2 = nn.LSTM(32,64,batch_first=True)
        self.output = nn.Linear(64,3)
        # Keras 기본 초기화 방식: 입력 Glorot, 순환 orthogonal, forget gate bias=1.
        # PyTorch의 추가 recurrent bias는 0에 고정해 Keras와 같은 학습 파라미터 수를 유지합니다.
        for layer in (self.encoder1,self.encoder2,self.decoder1,self.decoder2):
            nn.init.xavier_uniform_(layer.weight_ih_l0)
            nn.init.orthogonal_(layer.weight_hh_l0)
            nn.init.zeros_(layer.bias_ih_l0)
            with torch.no_grad(): layer.bias_ih_l0[layer.hidden_size:2*layer.hidden_size].fill_(1)
            nn.init.zeros_(layer.bias_hh_l0)
            layer.bias_hh_l0.requires_grad_(False)
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.zeros_(self.output.bias)
    def forward(self,x):
        x,_ = self.encoder1(x)
        _,(hidden,_) = self.encoder2(x)
        x = hidden[-1].unsqueeze(1).repeat(1,self.sequence,1)
        x,_ = self.decoder1(x)
        x,_ = self.decoder2(x)
        return self.output(x)

    @torch.inference_mode()
    def predict(self,x,verbose=0):
        self.eval()
        return np.concatenate([self(torch.tensor(x[i:i+32],dtype=torch.float32,device=device)).cpu().numpy()
                               for i in range(0,len(x),32)])
# PyTorch 기본 Adam과 Keras Adam은 epsilon의 위치가 달라 가이드 계산식을 명시합니다.
class KerasAdam(torch.optim.Optimizer):
    def __init__(self,params,lr=.001):
        super().__init__(params,dict(lr=lr,beta1=.9,beta2=.999,eps=1e-7,step=0))
    @torch.no_grad()
    def step(self,closure=None):
        for group in self.param_groups:
            group["step"] += 1
            anchor = group["params"][0]
            b1,b2 = [torch.tensor(group[k],dtype=anchor.dtype,device=anchor.device) for k in ("beta1","beta2")]
            alpha = torch.tensor(group["lr"],dtype=anchor.dtype,device=anchor.device)*(1-b2.pow(group["step"])).sqrt()/(1-b1.pow(group["step"]))
            for p in group["params"]:
                if p.grad is None: continue
                if not self.state[p]:
                    self.state[p]["m"] = torch.zeros_like(p)
                    self.state[p]["v"] = torch.zeros_like(p)
                m,v = self.state[p]["m"],self.state[p]["v"]
                m.add_((p.grad-m)*(1-b1))
                v.add_((p.grad.square()-v)*(1-b2))
                p.add_(-alpha*m/(v.sqrt()+group["eps"]))
def train_model(data, cfg, output):
    model = LSTMAutoencoder(cfg.sequence).to(device)
    print(model)
    print("학습 파라미터 수:", sum(p.numel() for p in model.parameters() if p.requires_grad))
    (output/"model_summary.txt").write_text(str(model),encoding="utf-8")
    optimizer = KerasAdam([p for p in model.parameters() if p.requires_grad],
                          lr=float(np.float32(cfg.learning_rate)))
    generator = torch.Generator().manual_seed(cfg.seed)
    train_loader = DataLoader(TensorDataset(torch.tensor(data["X_train"],dtype=torch.float32)),
        batch_size=cfg.batch_size, shuffle=True, generator=generator, num_workers=0)
    valid_loader = DataLoader(TensorDataset(torch.tensor(data["X_valid_0"],dtype=torch.float32)),
        batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    loss_fn = nn.MSELoss()
    best_loss = lr_best = float("inf")
    best_epoch = early_wait = lr_wait = 0
    early_stopped = False
    rows=[]
    started=time.perf_counter()
    for epoch in range(cfg.epochs):
        epoch_started=time.perf_counter()
        current_lr=optimizer.param_groups[0]["lr"]
        model.train()
        total=0.0
        for (x,) in train_loader:
            x=x.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss=loss_fn(model(x),x)
            loss.backward()
            optimizer.step()
            total+=float(loss.detach())*len(x)
        train_loss=total/len(data["X_train"])
        model.eval()
        with torch.inference_mode():
            total=0.0
            for (x,) in valid_loader:
                x=x.to(device)
                total+=float(loss_fn(model(x),x))*len(x)
        valid_loss=total/len(data["X_valid_0"])
        # Keras ReduceLROnPlateau의 기본 min_delta=1e-4(절대 차이)를 사용합니다.
        if valid_loss < lr_best-1e-4:
            lr_best,lr_wait=valid_loss,0
        else:
            lr_wait+=1
            if lr_wait>=cfg.lr_patience:
                for group in optimizer.param_groups:
                    group["lr"]=float(np.float32(group["lr"]*cfg.lr_factor))
                lr_wait=0
        early_wait+=1
        if valid_loss+cfg.es_min_delta < best_loss:
            best_loss,best_epoch,early_wait=valid_loss,epoch,0
            best_weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if early_wait>=cfg.es_patience and epoch>0:
            early_stopped=True
            model.load_state_dict(best_weights)
        rows.append(dict(epoch=epoch,loss=train_loss,val_loss=valid_loss,lr=current_lr,
                         epoch_seconds=time.perf_counter()-epoch_started))
        if epoch==0 or (epoch+1)%10==0:
            print(f"Epoch {epoch+1}: loss={train_loss:.8f}, val_loss={valid_loss:.8f}",flush=True)
        if early_stopped: break
    model.load_state_dict(best_weights)
    history=pd.DataFrame(rows)
    history.to_csv(output/"history.csv",index=False)
    training=dict(seconds=time.perf_counter()-started,epochs_run=len(history),epoch_limit=cfg.epochs,
        early_stopped=early_stopped,best_val_loss=float(history.val_loss.min()),
        minimum_val_loss_epoch=int(history.val_loss.argmin()+1),early_stopping_best_val_loss=best_loss,
        final_weights="best normal validation weights (min_delta rule)", selected_epoch=best_epoch+1)
    torch.save({"model_state_dict":model.state_dict(),"config":asdict(cfg)},output/"model.pt")
    save_json(output/"training.json",training)
    plt.figure(figsize=(10,5))
    plt.plot(history.loss,label="train loss")
    plt.plot(history.val_loss,label="valid loss")
    plt.xlabel("Epoch (zero based)"); plt.ylabel("MSE loss"); plt.legend()
    save_plot(output/"loss.png")
    return model,training

## 8. 평가 함수 — 마지막 시점 MSE, Valid 임계값, Test 고정

In [ ]:
%%guide
def score_arrays(model, x):
    reconstructed = model.predict(x)
    return np.mean((x[:, -1, :] - reconstructed[:, -1, :]) ** 2, axis=1)

def choose_f1_threshold(y, scores):
    p, r, thresholds = metrics.precision_recall_curve(y, scores)
    f1 = 2*p[:-1]*r[:-1] / np.maximum(p[:-1]+r[:-1], 1e-12)
    # 동률이면 가장 낮은 임계값 선택. PR curve와 실제 판정 모두 >= 사용.
    index = int(np.argmax(f1))
    return float(thresholds[index]), pd.DataFrame(dict(threshold=thresholds,
        precision=p[:-1], recall=r[:-1], f1=f1))

def metric_row(y, scores, threshold):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = metrics.confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(accuracy=metrics.accuracy_score(y, pred),
        precision=metrics.precision_score(y, pred, zero_division=0),
        recall=metrics.recall_score(y, pred, zero_division=0),
        f1=metrics.f1_score(y, pred, zero_division=0),
        average_precision=metrics.average_precision_score(y, scores),
        roc_auc=metrics.roc_auc_score(y, scores),
        false_positive_rate=float(fp/(tn+fp)), miss_rate=float(fn/(tp+fn)),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def evaluate_window(model, data, manifest, folder):
    valid_scores = score_arrays(model, data["X_valid"])
    threshold, curve = choose_f1_threshold(data["Y_valid"], valid_scores)
    curve.to_csv(folder/"threshold_curve.csv", index=False)
    result = dict(threshold=threshold)
    for split in ["valid", "test"]:
        scores = valid_scores if split == "valid" else score_arrays(model, data["X_test"])
        y = data["Y_"+split]
        result.update({split+"_"+k: v for k, v in metric_row(y, scores, threshold).items()})
        rows = manifest[manifest.split == split].copy()
        assert np.array_equal(rows.label.to_numpy(), y)
        rows["score"] = scores
        rows["prediction"] = (scores >= threshold).astype(int)
        rows["threshold"] = threshold
        rows.to_csv(folder/(split+"_predictions.csv"), index=False)
    return result

## 9. 반복 실험 실행
완료된 조합은 저장 결과를 재사용합니다. 중단된 조합은 처음부터 재학습합니다. 매 조합마다 seed·모델·optimizer를 초기화합니다. 결과 폴더가 남아 있는 런타임에서만 이어서 실행할 수 있습니다.

In [ ]:
%%guide
import gc
all_results = []
common_keys = None
for window in WINDOW_SIZES:
    current = dict(CONFIG, sequence=window, epochs=EXPERIMENT_EPOCHS)
    window_data, window_manifest = prepare_aligned(normal, outlier, Config(**current), ALIGN_WINDOW)
    keys = window_manifest[["split", "source", "input_end_row", "label_row", "label"]].reset_index(drop=True)
    if common_keys is None: common_keys = keys
    else: assert common_keys.equals(keys), "비교할 시점이 일치하지 않습니다."
    for seed in SEEDS:
        run_cfg = Config(**dict(current, seed=seed))
        folder = EXPERIMENT_DIR / ("w%03d_s%d" % (window, seed))
        folder.mkdir(parents=True, exist_ok=True)
        completed = folder / "result.json"
        if completed.exists():
            row = json.loads(completed.read_text())
            print("완료 결과 재사용:", folder.name)
        else:
            random.seed(seed); np.random.seed(seed)
            torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
            save_json(folder/"config.json", asdict(run_cfg))
            window_manifest.to_csv(folder/"split_manifest.csv", index=False)
            joblib.dump(window_data["scaler"], folder/"scaler.joblib")
            if device.type == "cuda":
                torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
            print("실험 시작:", folder.name, flush=True)
            run_model, training = train_model(window_data, run_cfg, folder)
            if device.type == "cuda": torch.cuda.synchronize()
            row = dict(window=window, seed=seed, epochs_run=training["epochs_run"],
                selected_epoch=training["selected_epoch"], train_seconds=training["seconds"],
                best_val_mse=training["best_val_loss"],
                gpu_peak_mb=torch.cuda.max_memory_allocated()/1024**2 if device.type == "cuda" else None,
                train_n=len(window_data["X_train"]), valid_n=len(window_data["X_valid"]),
                test_n=len(window_data["X_test"]))
            row.update(evaluate_window(run_model, window_data, window_manifest, folder))
            # 완료 파일은 모든 결과가 저장된 뒤 원자적으로 교체합니다.
            save_json(folder/"result.tmp", row)
            (folder/"result.tmp").replace(completed)
            del run_model
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
        all_results.append(row)
        pd.DataFrame(all_results).to_csv(EXPERIMENT_DIR/"runs.csv", index=False)
    del window_data
results = pd.DataFrame(all_results)
display(results)

## 10. 비교표와 그래프
평균 Valid F1로 후보를 정렬합니다. 동률이면 평균 Valid AP가 높은 후보, 다음으로 작은 window를 선택합니다. Test는 선택 완료 후 확인하는 최종 보고용입니다. seed 표준편차는 학습 변동성이고, 독립 고장 사건에 대한 신뢰구간이 아닙니다. seed가 1개이면 표준편차는 계산할 수 없어 NaN입니다.

In [ ]:
%%guide
columns = ["valid_f1", "valid_average_precision", "test_f1", "test_precision", "test_recall",
           "test_average_precision", "test_roc_auc", "test_false_positive_rate", "test_miss_rate",
           "train_seconds", "epochs_run", "gpu_peak_mb"]
summary = results.groupby("window")[columns].agg(["mean", "std"])
summary.columns = [a+"_"+b for a,b in summary.columns]
summary = summary.reset_index().sort_values(
    ["valid_f1_mean", "valid_average_precision_mean", "window"], ascending=[False, False, True])
summary.to_csv(EXPERIMENT_DIR/"summary.csv", index=False, encoding="utf-8-sig")
chosen = int(summary.iloc[0].window)
save_json(EXPERIMENT_DIR/"selection.json", dict(window=chosen,
    rule="mean valid F1, then mean valid AP, then smaller window"))
print("Valid 기준 선택 window:", chosen)
display(summary)
ordered = summary.sort_values("window")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, names in zip(axes, [["valid_f1", "test_f1"], ["test_recall", "test_false_positive_rate"], ["train_seconds"]]):
    for name in names:
        ax.errorbar(ordered.window, ordered[name+"_mean"],
            yerr=ordered[name+"_std"].fillna(0), marker="o", capsize=4, label=name)
    ax.set_xlabel("Window (rows)"); ax.legend(); ax.grid(alpha=.25)
axes[0].set_ylabel("Score"); axes[1].set_ylabel("Rate"); axes[2].set_ylabel("Seconds")
save_plot(EXPERIMENT_DIR/"comparison.png")
if 20 in WINDOW_SIZES:
    baseline = results[results.window == 20].set_index("seed")
    pairs = results.copy()
    for name in ["valid_f1", "test_f1", "test_recall", "test_false_positive_rate"]:
        pairs[name+"_delta_vs_20_pp"] = 100*(pairs[name]-pairs.seed.map(baseline[name]))
    pairs.to_csv(EXPERIMENT_DIR/"paired_differences.csv", index=False)
    display(pairs[["window", "seed"]+[c for c in pairs if c.endswith("_pp")]])

## 11. 결과 다운로드
`runs.csv`: 각 실험, `summary.csv`: 평균·표준편차, `paired_differences.csv`: 같은 seed의 window 20 대비 차이(%p), `comparison.png`: 비교 그래프. 각 조합 폴더에 모델·학습 이력·예측·분할 행을 저장합니다. ZIP 다운로드 셀은 Colab 기본 환경에서 실행합니다.

In [ ]:
# 별도 guide 커널에서 설정한 실제 경로를 전달받아 ZIP으로 묶습니다.
import zipfile
guide("", "Path('/content/window_experiment_path.txt').write_text(str(EXPERIMENT_DIR))")
experiment_folder = Path('/content/window_experiment_path.txt').read_text()
experiment_folder = Path(experiment_folder)
archive_path = experiment_folder.with_suffix('.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(experiment_folder.rglob('*')):
        if path.is_file(): archive.write(path, path.relative_to(experiment_folder))
files.download(str(archive_path))

## 해석 시 주의
- Window는 초가 아니라 행 수입니다. Timestamp 간격이 일정한지 확인한 뒤 실제 시간 범위로 해석하세요.
- 원본 offset=100은 마지막 입력 행에서 라벨까지 **101행**입니다. 이 규칙은 유지했습니다.
- 원본 두 CSV는 각각 정상/이상 단일 라벨입니다. 고장 전환점의 조기 탐지 시간을 검증하는 실험은 아닙니다.
- Valid/Test 사이에서 입력 행 중복을 제거했지만, 미래 라벨 시점까지 완전히 분리한 forecasting 검증은 아닙니다. 인접 시점의 상관도 남습니다. 본격 일반화 검증에는 별도 날짜·설비·고장 사건의 데이터가 필요합니다.
- 정상만으로 scaler를 fit하고, 모델은 정상 Train만 학습합니다. Test는 임계값 선정에 사용하지 않습니다.
- 표본 수가 바뀐 비교 프로토콜이므로 기존 가이드 수치와 직접 성능 우열을 판단하지 마세요.
- seed를 고정하고 deterministic 연산을 요청하지만 장치·버전 간 완전 동일 수치를 보장하지 않습니다. 지원하지 않는 연산 오류가 나면 조용히 비결정적 실행으로 바꾸지 말고 환경을 확인하세요.

참고: [PyTorch 재현성](https://docs.pytorch.org/docs/stable/notes/randomness.html), [scikit-learn 시계열 분할의 gap](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).